# Integración de agricultura, clima y edafología (2003–2025)

Esta notebook construye de forma reproducible el conjunto municipal para análisis y modelado. Utiliza los archivos agrícolas oficiales normalizados por el paquete, clima mensual de NASA POWER y polígonos edafológicos de INEGI.

La unidad agrícola original se conserva: la integración no debe crear ni eliminar registros.

In [ ]:
import geopandas as gpd
import pandas as pd

from proyecto_agronomia.config import (
    AGRICULTURA_RAW_DIR,
    ANIOS,
    ANIOS_MAPBIOMAS,
    EDAFOLOGIA_RAW_DIR,
    INEGI_INTERIM_DIR,
    MAPBIOMAS_INTERIM_DIR,
    MAPBIOMAS_RAW_DIR,
    NASA_POWER_PROCESSED_DIR,
    PROCESSED_DATA_DIR,
)
from proyecto_agronomia.datasets.mapbiomas import (
    CLAVE_NACIONAL,
    ruta_raster_mapbiomas,
)
from proyecto_agronomia.features import construir_historico_agricola #carga y normaliza los datos de agricultura
from proyecto_agronomia.processing.integracion_climatica import (
    construir_clima_mensual_ancho,
    crear_clave_municipal,
    integrar_agricultura_clima,
    resumir_clima_por_ciclo,
    validar_clima_mensual,
)
from proyecto_agronomia.processing.integracion_edafologica import (
    LLAVES_MASCARA,
    resumir_edafologia_municipal,
    resumir_edafologia_por_mascara,
)
from proyecto_agronomia.processing.integracion_mapbiomas import (
    asignar_mascara_mapbiomas,
    contar_suelo_por_mascara,
)
from proyecto_agronomia.processing.features_historicas import (
    COLUMNAS_HISTORICAS,
    agregar_historial_rendimiento,
)

RUTA_CLIMA_MENSUAL = (
    NASA_POWER_PROCESSED_DIR / "clima_municipal_mensual_2002_2025.csv"
)
RUTA_MUNICIPIOS = INEGI_INTERIM_DIR / "municipios_mexico_inegi.geojson"
RUTA_EDAFOLOGIA = (
    EDAFOLOGIA_RAW_DIR / "serie_iii" / "conjunto_de_datos"
    / "conj_nac_inf_edaf_esc_250k_ser_III_area.shp"
)
RUTA_EDAFOLOGIA_MUNICIPAL = PROCESSED_DATA_DIR / "edafologia_municipal.csv"
RUTA_EDAFOLOGIA_MASCARA = PROCESSED_DATA_DIR / "edafologia_mascara_mapbiomas.csv"
RUTA_AGRICULTURA_CLIMA = PROCESSED_DATA_DIR / "agricultura_clima_2003_2025.csv"
RUTA_FINAL = (
    PROCESSED_DATA_DIR / "agricultura_clima_edafologia_2003_2025.csv"
)
RUTA_FINAL_HISTORIAL = (
    PROCESSED_DATA_DIR / "agricultura_clima_edafologia_historial_2003_2025.csv"
)
RECALCULAR_EDAFOLOGIA_MUNICIPAL = False
RECALCULAR_EDAFOLOGIA_MASCARA = False
# Área mínima (ha) de la máscara MapBiomas para usar su composición de suelo;
# por debajo se usa la del municipio completo.
UMBRAL_MASCARA_HA = 1.0
pd.set_option("display.max_columns", None)

## Cargar y normalizar la producción agrícola

Los archivos `agricola_*.csv` se leen desde `data/raw/agricultura`. La normalización resuelve cambios históricos de columnas, nombres de cultivo, precios, separadores de miles y errores numéricos.

In [ ]:
df_agricultura = construir_historico_agricola(
    anios=ANIOS,
    directorio=AGRICULTURA_RAW_DIR,
)
df_agricultura = df_agricultura.drop(
    columns=[c for c in df_agricultura if c.startswith("Unnamed:")],
    errors="ignore",
)
df_agricultura = crear_clave_municipal(df_agricultura)
df_agricultura["Nomcicloproductivo"] = (
    df_agricultura["Nomcicloproductivo"].astype("string").str.strip()
)

columnas_agricolas_requeridas = {
    "Anio", "cvegeo", "Nomestado", "Nommunicipio",
    "Nomcultivo", "Nomcicloproductivo", "Nommodalidad",
    "Sembrada", "Cosechada", "Siniestrada",
    "Volumenproduccion", "Rendimiento",
}
faltantes = columnas_agricolas_requeridas - set(df_agricultura.columns)
assert not faltantes, f"Faltan columnas agrícolas: {sorted(faltantes)}"
assert set(df_agricultura["Anio"].unique()) == set(ANIOS)
assert df_agricultura["cvegeo"].str.fullmatch(r"\d{5}").all()

ciclos_validos = {"Otoño-Invierno", "Primavera-Verano", "Perennes"}
ciclos_encontrados = set(df_agricultura["Nomcicloproductivo"].dropna())
assert ciclos_encontrados <= ciclos_validos, (
    f"Ciclos no contemplados: {sorted(ciclos_encontrados-ciclos_validos)}"
)

print(f"Registros agrícolas: {len(df_agricultura):,}")
display(
    df_agricultura.groupby(["Anio", "Nomcicloproductivo"])
    .agg(
        registros=("cvegeo", "size"),
        municipios=("cvegeo", "nunique"),
        cultivos=("Nomcultivo", "nunique"),
    )
)

## Cargar y validar el clima mensual NASA POWER

La llave climática es `cvegeo + anio + mes`. El año 2002 se necesita para formar octubre–diciembre del ciclo Otoño–Invierno de 2003.

In [ ]:
if not RUTA_CLIMA_MENSUAL.exists():
    raise FileNotFoundError(
        f"No existe {RUTA_CLIMA_MENSUAL}. Ejecuta descarga_clima_nasa.ipynb."
    )

columnas_clima = [
    "cvegeo", "anio", "mes", "temperatura_media_mes_c",
    "precipitacion_mes_mm", "dias_esperados",
    "dias_temperatura", "dias_precipitacion",
]
df_clima_mensual = pd.read_csv(
    RUTA_CLIMA_MENSUAL,
    usecols=columnas_clima,
    dtype={"cvegeo": "string"},
)
df_clima_mensual = validar_clima_mensual(df_clima_mensual)
print(f"Registros climáticos mensuales: {len(df_clima_mensual):,}")
display(
    df_clima_mensual.groupby("anio").agg(
        municipios=("cvegeo", "nunique"),
        meses=("mes", "count"),
        temperatura_nulos=("temperatura_media_mes_c", lambda s: s.isna().sum()),
        precipitacion_nulos=("precipitacion_mes_mm", lambda s: s.isna().sum()),
    )
)

## Construir características climáticas

Se conservan los meses como columnas y se calculan ventanas compatibles con los ciclos productivos:

- Otoño–Invierno: octubre–diciembre anterior y enero–marzo actual.
- Primavera–Verano: abril–septiembre.
- Perennes: enero–diciembre.

In [ ]:
df_clima_ancho = construir_clima_mensual_ancho(df_clima_mensual)
df_clima_ciclos = resumir_clima_por_ciclo(df_clima_mensual)
df_clima_ciclos = df_clima_ciclos.loc[df_clima_ciclos["anio"].isin(ANIOS)]

assert not df_clima_ancho.duplicated(["cvegeo", "anio"]).any()
assert not df_clima_ciclos.duplicated(
    ["cvegeo", "anio", "Nomcicloproductivo"]
).any()
assert df_clima_ciclos[[
    "temperatura_media_ciclo_c", "precipitacion_ciclo_mm"
]].notna().all().all()

display(
    df_clima_ciclos.groupby(["anio", "Nomcicloproductivo"])
    .agg(
        municipios=("cvegeo", "nunique"),
        dias_min=("dias_clima_ciclo", "min"),
        dias_max=("dias_clima_ciclo", "max"),
    )
)

## Integrar agricultura y clima

Las uniones son `many_to_one`; por lo tanto, el número de registros agrícolas debe permanecer sin cambios y toda observación debe recibir clima.

In [ ]:
df_agricultura_clima = integrar_agricultura_clima(
    df_agricultura,
    df_clima_ciclos,
    df_clima_ancho,
)
columnas_climaticas_finales = [
    columna for columna in df_agricultura_clima.columns
    if columna.startswith(("temperatura_", "precipitacion_"))
]
assert len(df_agricultura_clima) == len(df_agricultura)
assert df_agricultura_clima[columnas_climaticas_finales].notna().all().all()

resumen_cobertura = (
    df_agricultura_clima.groupby(["Anio", "Nomcicloproductivo"])
    .agg(
        registros=("cvegeo", "size"),
        municipios=("cvegeo", "nunique"),
        temperatura=("temperatura_media_ciclo_c", "count"),
        precipitacion=("precipitacion_ciclo_mm", "count"),
    )
)
resumen_cobertura["cobertura_pct"] = (
    100 * resumen_cobertura["temperatura"] / resumen_cobertura["registros"]
).round(2)
assert resumen_cobertura["cobertura_pct"].eq(100).all()
display(resumen_cobertura)

RUTA_AGRICULTURA_CLIMA.parent.mkdir(parents=True, exist_ok=True)
df_agricultura_clima.to_csv(
    RUTA_AGRICULTURA_CLIMA, index=False, encoding="utf-8"
)
print(f"Dataset agricultura-clima: {RUTA_AGRICULTURA_CLIMA}")
print(f"Dimensiones: {df_agricultura_clima.shape}")

## Construir o cargar el resumen edafológico municipal

La intersección espacial puede tardar. Una vez generada, `edafologia_municipal.csv` se reutiliza. Cambia `RECALCULAR_EDAFOLOGIA_MUNICIPAL` a `True` solamente para reconstruirla. Si la caché es de una versión anterior (sin columnas `suelo_pct_*` o con columnas de fases), se reconstruye automáticamente.

Además del grupo de suelo dominante y la textura dominante, el resumen incluye la **composición edafológica**: una columna `suelo_pct_<código>` por cada grupo de suelo WRB con el porcentaje del área municipal que ocupa (0 si no está presente). `SA` (suelo cubierto por agua) se trata como un grupo más. Las fases física y química no se incluyen.

Este resumen municipal ya no es la fuente principal del suelo: se usa como **respaldo** del filtro MapBiomas de la siguiente sección.

In [ ]:
def _cache_edafologia_vigente(ruta):
    """La caché es válida si ya trae la composición y no trae fases."""
    if not ruta.exists():
        return False
    columnas = pd.read_csv(ruta, nrows=0).columns
    tiene_composicion = any(c.startswith("suelo_pct_") for c in columnas)
    tiene_fases = any(c.startswith("fase_") for c in columnas)
    return tiene_composicion and not tiene_fases


if _cache_edafologia_vigente(RUTA_EDAFOLOGIA_MUNICIPAL) and not RECALCULAR_EDAFOLOGIA_MUNICIPAL:
    df_edafologia_municipal = pd.read_csv(
        RUTA_EDAFOLOGIA_MUNICIPAL, dtype={"cvegeo": "string"}
    )
    df_edafologia_municipal["cvegeo"] = (
        df_edafologia_municipal["cvegeo"].str.zfill(5)
    )
else:
    for ruta in [RUTA_MUNICIPIOS, RUTA_EDAFOLOGIA]:
        if not ruta.exists():
            raise FileNotFoundError(f"No se encontró el archivo requerido: {ruta}")
    municipios = gpd.read_file(RUTA_MUNICIPIOS)
    edafologia = gpd.read_file(RUTA_EDAFOLOGIA)
    df_edafologia_municipal = resumir_edafologia_municipal(
        municipios, edafologia
    )
    df_edafologia_municipal.to_csv(
        RUTA_EDAFOLOGIA_MUNICIPAL, index=False, encoding="utf-8"
    )

COLUMNAS_COMPOSICION_SUELO = [
    c for c in df_edafologia_municipal.columns if c.startswith("suelo_pct_")
]
assert not df_edafologia_municipal["cvegeo"].duplicated().any()
print(f"Municipios edafológicos: {len(df_edafologia_municipal):,}")
print(f"Columnas de composición de suelo: {len(COLUMNAS_COMPOSICION_SUELO)}")
display(df_edafologia_municipal.head())

## Filtrar el suelo por la cobertura agrícola de MapBiomas

El municipio completo incluye bosques, matorrales, zonas urbanas, etc. Para aproximar mejor el suelo donde se siembra, se usa la cobertura anual de **MapBiomas México (Colección 1, 30 m, CC BY 4.0)**. Cada registro agrícola se asigna a una máscara según su tipo de cultivo:

| Máscara | Clase MapBiomas | Registros SIAP |
|---|---|---|
| `anual` | 19 Cultivo anual | ciclos Primavera–Verano y Otoño–Invierno |
| `perenne` | 36 Cultivo perenne | ciclo Perennes |
| `pastizal` | 15 Pastizal cultivado e inducido | pastos y zacates (`Pastos y praderas`, `Pasto tapete`, semillas de pasto/zacate), sin importar el ciclo |

Todos los cultivos del mismo tipo, municipio y año comparten los mismos datos edafológicos. Como el clima, esto es una **aproximación**: no se exige que el área de la máscara coincida con la superficie sembrada de SIAP. MapBiomas mide la tierra agrícola, incluido el barbecho, y SIAP lo sembrado; además, una parcela puede tener varios cultivos en el año.

- Un píxel pertenece a la máscara del año *t* si tuvo la clase en *t−1*, *t* o *t+1*. Así se absorbe el ruido interanual de la clasificación y los barbechos. Por eso se descarga desde 2002; en 2025 la ventana queda en 2024–2025.
- La composición de suelo (`suelo_pct_*`, dominantes, textura) se calcula como % del área de la máscara en el municipio y el año.
- **Respaldos** cuando la máscara propia mide menos de `UMBRAL_MASCARA_HA`:
  - perennes → máscara `agricola` (clases 19+36), porque MapBiomas clasifica como cultivo anual buena parte de la caña, los cítricos, la alfalfa y la vid;
  - si tampoco hay, o en anuales y pastizales, el resumen municipal.
- `suelo_fuente` indica qué se usó (`mascara`, `agricola` o `municipal`) y `area_mascara_ha` el tamaño de la máscara propia.
- La clase 21 *Mosaico de usos* no se usa: prácticamente no aparece en la Colección 1.

Los rasters se descargan con `descarga_mapbiomas.ipynb` en `data/raw/mapbiomas/00/<año>.tif`. El cruce raster × municipios × suelos se guarda por estado en `data/interim/mapbiomas/suelo_por_mascara/` y se reanuda si se interrumpe (unos 15 minutos la primera vez).

In [ ]:
df_agricultura_clima["mascara"] = asignar_mascara_mapbiomas(
    df_agricultura_clima["Nomcultivo"], df_agricultura_clima["Nomcicloproductivo"]
)
assert df_agricultura_clima["mascara"].notna().all()
display(
    df_agricultura_clima.groupby(["mascara", "Nomcicloproductivo"], observed=True)
    .agg(registros=("cvegeo", "size"), sembrada_ha=("Sembrada", "sum"))
    .round(0)
)

llaves_agricolas = df_agricultura_clima[LLAVES_MASCARA].drop_duplicates()


RUTA_PARAMETROS_CRUCE = MAPBIOMAS_INTERIM_DIR / "suelo_por_mascara" / "parametros.json"


def _cache_mascara_vigente(ruta):
    """Vigente si es más nueva que el cruce raster (cuyos parámetros cambian al
    cambiar máscaras, años o ventana) y trae las columnas actuales."""
    if not (ruta.exists() and RUTA_PARAMETROS_CRUCE.exists()):
        return False
    if ruta.stat().st_mtime < RUTA_PARAMETROS_CRUCE.stat().st_mtime:
        return False
    columnas = set(pd.read_csv(ruta, nrows=0).columns)
    return {"mascara", "suelo_fuente", "area_mascara_ha"} <= columnas


if _cache_mascara_vigente(RUTA_EDAFOLOGIA_MASCARA) and not RECALCULAR_EDAFOLOGIA_MASCARA:
    df_edafologia = pd.read_csv(
        RUTA_EDAFOLOGIA_MASCARA, dtype={"cvegeo": "string", "mascara": "string"}
    )
    df_edafologia["cvegeo"] = df_edafologia["cvegeo"].str.zfill(5)
else:
    faltan_rasters = [
        anio for anio in ANIOS_MAPBIOMAS
        if not ruta_raster_mapbiomas(CLAVE_NACIONAL, anio, MAPBIOMAS_RAW_DIR).exists()
    ]
    if faltan_rasters:
        raise FileNotFoundError(
            f"Faltan los rasters MapBiomas de {faltan_rasters}. "
            "Ejecuta primero notebooks/descarga_mapbiomas.ipynb."
        )
    municipios = gpd.read_file(RUTA_MUNICIPIOS)
    edafologia = gpd.read_file(RUTA_EDAFOLOGIA, columns=["Grupo1", "Clase_tex"])
    conteos_mascara = contar_suelo_por_mascara(
        municipios, edafologia, anios=ANIOS, recalcular=RECALCULAR_EDAFOLOGIA_MASCARA
    )
    df_edafologia = resumir_edafologia_por_mascara(
        conteos_mascara,
        df_edafologia_municipal,
        llaves_agricolas,
        umbral_ha=UMBRAL_MASCARA_HA,
    )
    df_edafologia.to_csv(RUTA_EDAFOLOGIA_MASCARA, index=False, encoding="utf-8")

COLUMNAS_COMPOSICION_SUELO = [
    c for c in df_edafologia.columns if c.startswith("suelo_pct_")
]
assert not df_edafologia.duplicated(LLAVES_MASCARA).any()
faltan_llaves = llaves_agricolas.merge(
    df_edafologia[LLAVES_MASCARA], how="left", indicator=True
)["_merge"].eq("left_only")
assert not faltan_llaves.any(), f"{faltan_llaves.sum()} llaves agrícolas sin resumen."

print(f"Combinaciones municipio-año-máscara: {len(df_edafologia):,}")
print("Fuente del suelo (% de combinaciones):")
display(
    pd.crosstab(df_edafologia["mascara"], df_edafologia["suelo_fuente"], normalize="index")
    .mul(100).round(1)
)
display(
    df_edafologia.groupby("mascara")["area_mascara_ha"]
    .describe(percentiles=[0.1, 0.5, 0.9]).round(1)
)

## Integrar suelo y guardar el dataset final

El resumen edafológico tiene una fila por municipio, año y máscara (`cvegeo`, `Anio`, `mascara`), así que la unión tampoco debe multiplicar registros agrícolas.

In [ ]:
df_final = df_agricultura_clima.merge(
    df_edafologia,
    on=LLAVES_MASCARA,
    how="left",
    validate="many_to_one",
)
assert len(df_final) == len(df_agricultura), (
    "La integración edafológica alteró el número de registros."
)
columnas_suelo = [
    "grupo_suelo_dominante", "nombre_suelo_wrb",
    "grupo_suelo_dominante_pct", "grupos_suelo_n",
    "textura_dominante", "area_mascara_ha", "suelo_fuente",
]
faltantes_suelo = df_final["grupo_suelo_dominante"].isna()
if faltantes_suelo.any():
    claves_faltantes = sorted(df_final.loc[faltantes_suelo, "cvegeo"].unique())
    raise ValueError(
        f"Hay {len(claves_faltantes)} municipios agrícolas sin suelo: "
        f"{claves_faltantes[:20]}"
    )

# La composición se expresa sobre el área de la máscara (o del municipio en el
# respaldo), por lo que no puede superar 100 %. Puede quedar por debajo donde
# la capa edafológica no cubre lagunas o mar.
assert df_final[COLUMNAS_COMPOSICION_SUELO].notna().all().all()
suma_composicion = df_edafologia[COLUMNAS_COMPOSICION_SUELO].sum(axis=1)
assert suma_composicion.le(100.5).all(), suma_composicion.describe()
print("Cobertura edafológica (% del área de referencia):")
display(
    suma_composicion.groupby(df_edafologia["suelo_fuente"]).describe().round(2)
)
print(f"Combinaciones con cobertura < 95 %: {suma_composicion.lt(95).sum():,}")

print("Registros agrícolas por fuente de suelo (%):")
display(
    pd.crosstab(df_final["mascara"], df_final["suelo_fuente"], normalize="index")
    .mul(100).round(1)
)

df_final.to_csv(RUTA_FINAL, index=False, encoding="utf-8")
print(f"Dataset final: {RUTA_FINAL}")
print(f"Registros: {len(df_final):,}")
print(f"Columnas: {len(df_final.columns):,}")
print(f"Periodo: {df_final['Anio'].min()}–{df_final['Anio'].max()}")
display(
    df_final[[
        "Anio", "cvegeo", "Nomestado", "Nommunicipio",
        "Nomcultivo", "Nomcicloproductivo", "mascara",
        "temperatura_media_ciclo_c", "precipitacion_ciclo_mm",
        *columnas_suelo, *COLUMNAS_COMPOSICION_SUELO,
    ]].head()
)

In [ ]:
df_final.info()

## Agregar estabilidad histórica sin fuga temporal

Para cada municipio, cultivo, ciclo, modalidad y unidad, las características del año actual utilizan exclusivamente rendimientos de años anteriores. Los primeros años permanecen sin historial suficiente.

In [ ]:
df_final_historial = agregar_historial_rendimiento(df_final, minimo_periodos=3)
assert len(df_final_historial) == len(df_final)
assert df_final_historial.loc[
    df_final_historial['Anio'].eq(df_final_historial['Anio'].min()),
    'rendimiento_hist_media',
].isna().all()
df_final_historial.to_csv(RUTA_FINAL_HISTORIAL, index=False, encoding='utf-8')
print(f'Dataset con historial: {RUTA_FINAL_HISTORIAL}')
display(
    df_final_historial[[
        'Anio', 'cvegeo', 'Nomcultivo', 'Nomcicloproductivo',
        'Nommodalidad', *COLUMNAS_HISTORICAS,
    ]].loc[lambda tabla: tabla['rendimiento_hist_media'].notna()].head()
)

## Recargar y validar el producto

Esta última lectura verifica que el CSV persistido conserva la llave municipal, el periodo y las variables esenciales.

In [ ]:
df_verificacion = pd.read_csv(
    RUTA_FINAL_HISTORIAL, dtype={"cvegeo": "string"}, low_memory=False
)
df_verificacion["cvegeo"] = df_verificacion["cvegeo"].str.zfill(5)
assert len(df_verificacion) == len(df_final)
assert set(df_verificacion["Anio"].unique()) == set(ANIOS)
assert df_verificacion["cvegeo"].str.fullmatch(r"\d{5}").all()
assert df_verificacion[[
    "temperatura_media_ciclo_c", "precipitacion_ciclo_mm",
    "grupo_suelo_dominante",
]].notna().all().all()
assert set(COLUMNAS_HISTORICAS).issubset(df_verificacion.columns)
display(pd.DataFrame({
    "archivo": [str(RUTA_FINAL_HISTORIAL)],
    "registros": [len(df_verificacion)],
    "columnas": [len(df_verificacion.columns)],
    "municipios": [df_verificacion["cvegeo"].nunique()],
    "cultivos": [df_verificacion["Nomcultivo"].nunique()],
    "anio_inicial": [df_verificacion["Anio"].min()],
    "anio_final": [df_verificacion["Anio"].max()],
}))